### Step 1: keep only stations with real coverage

We already verified the raw station list. The important point is that not every PM2.5 monitor is good enough.

We will use a measurable rule:

- calculate actual coverage_days = (last_reading - first_reading).days
- keep only stations where coverage_days >= 730

This is the minimum quality gate for version one.

A station with only a few months of data is too weak for a stable forecast.

In [1]:
import pandas as pd

stations = pd.read_csv("../data/raw/stations.csv")

stations["first_reading"] = pd.to_datetime(stations["first_reading"], errors="coerce")
stations["last_reading"] = pd.to_datetime(stations["last_reading"], errors="coerce")

stations["coverage_days"] = (
    stations["last_reading"] - stations["first_reading"]
).dt.days

valid_stations = stations[
    stations["coverage_days"] >= 730
].copy()

valid_stations = valid_stations.sort_values(
    by=["coverage_days", "last_reading"],
    ascending=[False, False]
).reset_index(drop=True)

print("Raw stations:", len(stations))
print("Valid stations:", len(valid_stations))
print("Unique valid locations:", valid_stations["location_id"].nunique())

display(valid_stations.head(10))

Raw stations: 147
Valid stations: 50
Unique valid locations: 50


,location_id,sensor_id,name,latitude,longitude,provider,first_reading,last_reading,coverage_days
0,8118,23534,New Delhi,28.635760,77.224450,AirNow,2016-11-10 00:30:00+05:30,2026-09-16 16:00:00+05:30,3597.0
1,301,1166,"Vikas Sadan, Gurugram - HSPCB",28.450124,77.026305,CPCB,2016-03-25 13:30:00+05:30,2022-10-31 07:30:00+05:30,2410.0
2,5610,14860,"North Campus, DU, Delhi - IMD",28.657381,77.158545,CPCB,2018-03-09 11:30:00+05:30,2022-10-31 07:30:00+05:30,1696.0
3,5617,14930,"Sector- 16A, Faridabad - HSPCB",28.408842,77.309908,caaqm,2018-03-09 11:30:00+05:30,2022-10-31 07:30:00+05:30,1696.0
4,5622,14922,"NSIT Dwarka, Delhi - CPCB",28.609090,77.032541,CPCB,2018-03-09 11:30:00+05:30,2022-10-31 07:30:00+05:30,1696.0
5,5630,14959,"Shadipur, Delhi - CPCB",28.651478,77.147311,CPCB,2018-03-09 11:30:00+05:30,2022-10-31 07:30:00+05:30,1696.0
6,5665,15239,"Vasundhara, Ghaziabad - UPPCB",28.660335,77.357256,CPCB,2018-03-09 11:30:00+05:30,2022-10-31 07:30:00+05:30,1696.0
7,5626,14985,"DTU, New Delhi - CPCB",28.750050,77.111261,CPCB,2018-03-09 11:00:00+05:30,2022-10-31 07:15:00+05:30,1696.0
8,5627,14935,"CRRI Mathura Road, New Delhi - IMD",28.551201,77.273574,CPCB,2018-03-09 11:00:00+05:30,2022-10-31 06:45:00+05:30,1696.0
9,5509,14521,"Anand Vihar, Delhi - DPCC",28.647622,77.315809,caaqm,2018-03-09 09:00:00+05:30,2022-10-31 06:00:00+05:30,1696.0


### What to read from this output

This tells us:

- the project is filtering out weak stations
- the valid pool is smaller and more trustworthy
- we now have the minimum-quality station list for the first model

This is the first hard quality gate.

### Step 2: use only the most recent 4 years

Now we choose the actual training time window.

For version one, we will use the most recent 4 years from the valid station pool.

Why:

- it matches the current Delhi environment
- it avoids very old data with different behavior
- it still gives enough history for seasonal learning

In [ ]:
latest_reading = valid_stations["last_reading"].max()  # scan the whole column, keep only the single most recent date
print(latest_reading)  # show that one date
print(type(latest_reading))  # confirm it's one Timestamp, not a list/column

# latest_reading is a single Timestamp, not a collection -> count rows that match it
stations_on_latest_date = valid_stations[valid_stations["last_reading"] == latest_reading]  # rows: last_reading equals the max exactly
print("Stations with that exact latest reading:", len(stations_on_latest_date))  # how many rows matched

window_start = latest_reading - pd.DateOffset(years=4)  # go back 4 calendar years from the latest date

print("Latest available reading:", latest_reading)
print("Window start:", window_start)

recent_valid_stations = valid_stations[
    valid_stations["last_reading"] >= window_start  # keep rows whose last_reading falls on/after window_start
].copy()  # .copy() avoids a "view vs copy" warning when we edit this subset later

print("Stations in recent 4-year window:", len(recent_valid_stations))  # row count after the date filter
print("Unique locations in recent window:", recent_valid_stations["location_id"].nunique())  # distinct station IDs (no duplicates)

display(recent_valid_stations.head(10))  # preview first 10 rows of the filtered table

2026-09-16 16:30:00+05:30
<class 'pandas._libs.tslibs.timestamps.Timestamp'>
Stations with that exact latest reading: 1
Latest available reading: 2026-09-16 16:30:00+05:30
Window start: 2022-09-16 16:30:00+05:30
Stations in recent 4-year window: 44
Unique locations in recent window: 44


,location_id,sensor_id,name,latitude,longitude,provider,first_reading,last_reading,coverage_days
0,8118,23534,New Delhi,28.635760,77.224450,AirNow,2016-11-10 00:30:00+05:30,2026-09-16 16:00:00+05:30,3597.0
1,301,1166,"Vikas Sadan, Gurugram - HSPCB",28.450124,77.026305,CPCB,2016-03-25 13:30:00+05:30,2022-10-31 07:30:00+05:30,2410.0
2,5610,14860,"North Campus, DU, Delhi - IMD",28.657381,77.158545,CPCB,2018-03-09 11:30:00+05:30,2022-10-31 07:30:00+05:30,1696.0
3,5617,14930,"Sector- 16A, Faridabad - HSPCB",28.408842,77.309908,caaqm,2018-03-09 11:30:00+05:30,2022-10-31 07:30:00+05:30,1696.0
4,5622,14922,"NSIT Dwarka, Delhi - CPCB",28.609090,77.032541,CPCB,2018-03-09 11:30:00+05:30,2022-10-31 07:30:00+05:30,1696.0
5,5630,14959,"Shadipur, Delhi - CPCB",28.651478,77.147311,CPCB,2018-03-09 11:30:00+05:30,2022-10-31 07:30:00+05:30,1696.0
6,5665,15239,"Vasundhara, Ghaziabad - UPPCB",28.660335,77.357256,CPCB,2018-03-09 11:30:00+05:30,2022-10-31 07:30:00+05:30,1696.0
7,5626,14985,"DTU, New Delhi - CPCB",28.750050,77.111261,CPCB,2018-03-09 11:00:00+05:30,2022-10-31 07:15:00+05:30,1696.0
8,5627,14935,"CRRI Mathura Road, New Delhi - IMD",28.551201,77.273574,CPCB,2018-03-09 11:00:00+05:30,2022-10-31 06:45:00+05:30,1696.0
9,5509,14521,"Anand Vihar, Delhi - DPCC",28.647622,77.315809,caaqm,2018-03-09 09:00:00+05:30,2022-10-31 06:00:00+05:30,1696.0


### What this means

This is the actual training-slice decision.

We are not using:

- all old stations
- all old dates
- every monitor that exists

We are using:

- only valid stations
- only the recent 4-year range
- the slice that is most relevant to the problem

### Step 3: save the filtered station list

We now save the filtered station list so the next notebook step can use it cleanly.

This makes the data pipeline reproducible.

In [7]:
from pathlib import Path

filtered_dir = Path("../data/processed")
filtered_dir.mkdir(parents=True, exist_ok=True)

filtered_stations_path = filtered_dir / "valid_stations_recent_4y.csv"
recent_valid_stations.to_csv(filtered_stations_path, index=False)

print("Saved filtered station list to:", filtered_stations_path)
print("Rows saved:", len(recent_valid_stations))
print("Columns saved:", list(recent_valid_stations.columns))

Saved filtered station list to: ../data/processed/valid_stations_recent_4y.csv
Rows saved: 44
Columns saved: ['location_id', 'sensor_id', 'name', 'latitude', 'longitude', 'provider', 'first_reading', 'last_reading', 'coverage_days']
